In [ ]:
score_configs = [
    ('Март', 'ZTAN57', '2026-03-05', '2026-03-12', '2026-03-26'),
    ('Апрель', 'ZTANTTT63', '2026-04-02', '2026-04-09', '2026-04-30')
]

bucket = 'cvm-current-customer-data'

scores = pd.concat(
    [
        (
            lambda audience:
            su.get_pandas_s3(
                prefix=(
                    'ml_platform/subscription/'
                    'cvm_subscription_propensity/permanent/'
                    f'model_predictions/{score_dt}/'
                    'subscription_propensity/score.parquet'
                ),
                bucket=bucket,
                engine='pyarrow'
            )
            .query('contact_id in @audience.contact_id')
            .assign(
                score_month=month,
                campaign_name=campaign,
                date_start=date_start,
                date_end=date_end
            )
        )(
            su.execute_custom_query_gp(f"""
                select distinct contact_id
                from cvm_sbx.giv_CVMB_24118_campaigns_audience
                where campaign_name = '{campaign}'
            """)
        )

        for month, campaign, score_dt, date_start, date_end
        in score_configs
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(scores['score'], errors='coerce')